# 03_Map_par mostrar

En este notebook usamos `folium` para mostrar la rejilla espacial que generamos en `02_agrupar_por_lat_lon.ipynb`.

- Cada punto corresponde a un cuadro de `lat | long`.
- El popup muestra la `especie` dominante y su `cantidad`.
- `folium` es ideal para mapas interactivos ligeros y para embellecer la presentación.


In [1]:
import os
import folium
from folium.plugins import MarkerCluster

try:
    import folium
    from folium.plugins import MarkerCluster
except ImportError as e:
    raise ImportError(
        "Folium no está instalado en este kernel. Ejecuta en terminal: python -m pip install folium"
    ) from e

print("folium version:", folium.__version__)


folium version: 0.20.0


In [2]:

from pyspark.sql import SparkSession

# 1. CONFIGURACIÓN JAVA
jdk_path = os.path.expanduser('~/.jdk17')

if os.path.isdir(jdk_path):
    os.environ['JAVA_HOME'] = jdk_path
    os.environ['PATH'] = os.path.join(jdk_path, 'bin') + os.pathsep + os.environ.get('PATH', '')
    print('JAVA_HOME configurado con éxito.')
else:
    print('No se encontró Java 17.')


JAVA_HOME configurado con éxito.


In [3]:
import os

print("Current dir:", os.getcwd())
print(os.listdir())

Current dir: /workspaces/bigData-Class/Notebook
['bono_varianza.py', 'ejercicio1.py', '03_Map_par mostrar.ipynb', 'ejercicio2.py', 'Notebook', 'ejercicio3.py', '01_read_filter_data.ipynb', 'biodiversidad_limpa.parquet', 'ejercicio5.py', 'votos_actas.txt', 'biodiversidad_clusters.parquet', 'datos_para_mapa_rejilla.parquet', 'map_reduce_votaciones.ipynb', 'resultados.txt', '02_agrupar_por_lat_lon.ipynb', 'biodiversidad_especies_por_punto.parquet', '0004764-260519110011954.csv', 'ejercicio4.py', 'votos_sueltos.txt']


In [4]:
import os
print(os.getcwd())
print(os.path.exists("datos_para_mapa_rejilla.parquet"))

/workspaces/bigData-Class/Notebook
True


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Biodiversidad") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df = spark.read.parquet("datos_para_mapa_rejilla.parquet")

print("Registros:", df.count())

pandas_df = df.toPandas()

spark.stop()

pandas_df.head()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 22:09:46 WARN Utils: Your hostname, codespaces-e48955, resolves to a loopback address: 127.0.0.1; using 10.0.13.103 instead (on interface eth0)
26/05/29 22:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 22:09:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Registros: 2306


,lat,long,especie,cantidad
0,11.1,-72.7,Vachellia macracantha,5316
1,10.1,-73.9,Astronium graveolens,1999
2,11.0,-72.7,unknown,1519
3,10.1,-73.8,Astronium graveolens,1443
4,10.8,-72.9,unknown,1382


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Biodiversidad") \
    .master("local[*]") \
    .getOrCreate()

parquet_path = "./datos_para_mapa_rejilla.parquet"

df = spark.read.parquet(parquet_path)

print(df.count())

df.show(5)

2306
+----+-----+--------------------+--------+
| lat| long|             especie|cantidad|
+----+-----+--------------------+--------+
|11.1|-72.7|Vachellia macraca...|    5316|
|10.1|-73.9|Astronium graveolens|    1999|
|11.0|-72.7|             unknown|    1519|
|10.1|-73.8|Astronium graveolens|    1443|
|10.8|-72.9|             unknown|    1382|
+----+-----+--------------------+--------+
only showing top 5 rows


In [7]:
import folium
from folium.plugins import MarkerCluster

if pandas_df.empty:
    raise ValueError("El DataFrame está vacío. Ejecuta primero la celda de carga de datos.")

center_lat = float(pandas_df["lat"].mean())
center_lon = float(pandas_df["long"].mean())
mapa = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles="OpenStreetMap")
marker_cluster = MarkerCluster(name="Hotspots por cuadro").add_to(mapa)

def marker_color(cantidad):
    if cantidad >= 1000:
        return "darkred"
    if cantidad >= 500:
        return "red"
    if cantidad >= 100:
        return "orange"
    return "blue"

for _, row in pandas_df.iterrows():
    popup = folium.Popup(
        f"<b>Especie:</b> {row['especie']}<br>"
        f"<b>Cantidad:</b> {int(row['cantidad'])}<br>"
        f"<b>Lat:</b> {row['lat']}<br>"
        f"<b>Lon:</b> {row['long']}",
        max_width=300,
    )
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=max(4, min(16, int(row['cantidad'] / 100))),
        color=marker_color(int(row['cantidad'])),
        fill=True,
        fill_color=marker_color(int(row['cantidad'])),
        fill_opacity=0.7,
        popup=popup,
    ).add_to(marker_cluster)

title_html = '<h3 align="center">Mapa de cuadros de biodiversidad (Folium)</h3>'
mapa.get_root().html.add_child(folium.Element(title_html))

output_path = os.path.join(os.getcwd(), "03_map_cuadros_folium.html")
mapa.save(output_path)
print("Mapa guardado en:", output_path)
mapa


Mapa guardado en: /workspaces/bigData-Class/Notebook/03_map_cuadros_folium.html


## ¿Qué puedes ajustar?

- Cambia el tamaño de los círculos en `radius=max(4, min(16, int(row["cantidad"] / 100)))` para que se vean mejor según tus datos.
- Cambia `round(..., 1)` en `02` por `round(..., 2)` para cuadros más pequeños.
- Si quieres etiquetas más grandes, puedes usar `folium.Marker` en lugar de `CircleMarker`.
